# N-gram mining: what skills is the taxonomy missing?

M1 found only ~24% of postings matched any taxonomy term. Two possible causes: we're **missing skills** (real skills not in our 75-term list) or **missing languages** (English-only taxonomy under-matching German/Spanish text). This notebook tackles the first.

It mines the most frequent 1-/2-/3-word phrases across every job description (all languages), drops the ones the taxonomy already knows, and ranks what's left by **document frequency** (distinct jobs containing the phrase). The output is a *candidate* list for human review — nothing is auto-promoted into `skills.yaml`.

In [1]:
import sys
from pathlib import Path

import pandas as pd

# The notebook lives in notebooks/; put the repo root on sys.path so
# `import jobskills` resolves no matter where Jupyter was launched from.
REPO_ROOT = Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

pd.set_option("display.max_colwidth", 80)

from jobskills.mine import candidates

## Candidate phrases

Phrases appearing in at least 10 distinct jobs and **not** already in the taxonomy, with a context snippet so each can be judged at a glance. Read this as a worklist: reject roles (`data analyst`) and filler (`about`), keep real missing skills.

In [2]:
cand = candidates(min_postings=10)
print(f"{len(cand)} candidate phrases (>=10 distinct jobs, not in taxonomy)")
cand.head(30)

2176 candidate phrases (>=10 distinct jobs, not in taxonomy)


## Reading the list

The ranking is dominated by **noise we expect to reject**, and that's fine — mining surfaces, the human filters:

- **Roles, not skills** — `data analyst`, `data scientist`, `engineer`. These are what we *searched for*, so of course they top the list. Not taxonomy material.
- **Generic filler** — `about`, `senior`, `global`, `join`, `world`. Common English, low signal.
- **Stopword leakage** — `du`, `unsere` (German), `re` (contraction residue from "we're"/"you're"). These tell us the stopword wall needs a few more entries.

**The genuine find: `ai`** — appears in ~230 distinct jobs and is a real skill *missing* from the taxonomy. That alone justifies the step: artificial intelligence is in high demand and we weren't counting it.

**On the language question:** the candidate list is overwhelmingly English even though ~60% of the corpus is German/Spanish. That's the *cross-language fragmentation* we predicted — a German skill phrase splits its count across DE surface forms and never rises high enough to surface here. So this notebook isolates the **coverage gap** (English skills we missed, like `ai`); the **language gap** is step 11b's job (DE/ES aliases), informed by re-mining once those are in.

## Production equivalent: `CountVectorizer`

The miner above is hand-rolled (`collections.Counter`) — built that way to *own* the mechanics. In real work you'd reach for scikit-learn's `CountVectorizer`, which does the same tokenize → n-gram → count in a few lines. The module keeps both as selectable backends; here we prove they produce the **identical** ranking.

The load-bearing equivalence: `binary=True` makes each matrix cell 0/1 (presence), so column sums *are* document frequencies — exactly what the hand-rolled per-document `set()` computes.

In [3]:
sk = candidates(min_postings=10, backend="sklearn")

hand = dict(zip(cand.ngram, cand.n_postings))
skl = dict(zip(sk.ngram, sk.n_postings))
assert hand == skl, "backends disagree"
print(f"hand-rolled and CountVectorizer agree on all {len(hand)} candidates "
      f"(counts identical).")

hand-rolled and CountVectorizer agree on all 2176 candidates (counts identical).
